In [1]:
import duckdb
import pandas as pd
from pathlib import Path

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')

MY_DATA_PATH = Path('/home/lc/m/working/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')

In [2]:
class SetUp:

    def __init__(self):
        self._setup_db()
        return
    
    def _setup_db(self):
        self.db = duckdb.connect()
        self.db.sql(f"ATTACH IF NOT EXISTS '{MY_DATABASE_FILE}' AS project")                     
        print(self.db.sql("SHOW ALL TABLES").df())
        return

In [3]:
class ETL(SetUp):

    def __init__(self):
        super().__init__()
        self.assemble_tables = {}
        return
    
    def sample_table(self):
        cols = ['Research_Profile', 'ACR', 'PUB', 'CIT', 'HCP', 'suma', 'coc', 'score', 'Group', 
                'author_id', 'orcid', 'fullname', 'works_count_endogenous', 'citations_endogenous', 'hca_endogenous', 
                'works_count_total', 'cited_by_count', 'hca_total', '2yr_mean_citedness', 'h_index', 'citations_total_oa']       
        sample = self.db.sql("SELECT * FROM project.sample_matched").df()[cols]
        sample = sample.sort_values(['Group', 'Research_Profile'], ascending=[False, True]).reset_index(drop=True)
        print(f'{sample.shape = }\n{sample.head()}')
        self.assemble_tables |= {'Domingo_matched': sample}
        return
    
    def works_table(self):
        # cols = [c.strip() for c in temp.split(" ") if c]
        cols = ['first_page', 'work_id', 'doi', 'title', 'publication_year', 'type', 
                'countries_distinct_count', 'institutions_distinct_count', 
                'fwci', 'cited_by_count', 'referenced_works_count', 
                'source_id', 'source_name', 'host_id', 'host_name']
        works = self.db.sql("SELECT * FROM project.works WHERE referenced_works_count > 0 and first_page != 'i'").df()[cols]
        works = works[works.referenced_works_count > 0].sort_values(['publication_year', 'fwci'], ascending=[True, False]).\
            drop(columns=['first_page']).reset_index(drop=True)
        print(f'{works.shape = }\n{works.head()}')
        self.assemble_tables |= {'works': works}
        return
    
    def authorships_table(self):
        sql = """SELECT a.work_id, author_id, author_name, orcid, institution_id, institution_name, country_code, a.type 
                    FROM project.works 
                    INNER JOIN project.authorships a
                    USING (work_id) 
                    WHERE referenced_works_count > 0 AND first_page != 'i'
                """
        authorships = self.db.sql(sql).df()
        print(f"{authorships.shape = }\n{authorships.head()}")
        self.assemble_tables |= {'authorships': authorships}
        return
    
    def references_table(self):
        sql = """
                SELECT c.work_id AS citer_id,
                        len(referenced_works) AS references_count,
                        referenced_works AS cited_id 
                    FROM project.cited c
                    INNER JOIN project.works w
                    USING (work_id)
                    WHERE referenced_works_count > 0 and first_page != 'i'
            """
        references = self.db.sql(sql).df()
        print(f'{references.shape = }\n{references.head()}')
        self.assemble_tables |= {'references': references}
        return
    
    def citations_table(self):
        sql = """
            SELECT cited_id,
                    count(citer_id) AS citations_count,
                    list(citer_id) AS citing_works 
                FROM
                    (SELECT work_id AS citer_id,
                            unnest(referenced_works) AS cited_id,
                        FROM project.cited) c
                LEFT JOIN project.works w
                ON c.cited_id = w.work_id
                WHERE referenced_works_count > 0 and first_page != 'i'
                GROUP BY cited_id
                ORDER BY citations_count DESC
            """
        citations = self.db.sql(sql).df()
        print(f'{citations.shape = }\n{citations.head()}')
        self.assemble_tables |= {'citations': citations}
        return

    def load_tables(self):
        for k, v in self.assemble_tables.items():
            with open(f'../DATA/tables_for_Domingo_{k}.csv', 'w') as writer:
                print(f'{k = } {v.shape = }\n{v.head()}')
                v.to_csv(writer, index=False)


In [4]:
def main():

    etl = ETL()
    etl.sample_table()
    etl.works_table()
    etl.authorships_table()
    etl.references_table()
    etl.citations_table()
    etl.load_tables()

    return

In [5]:
if __name__ == "__main__":
    main()
    print("DONE")

InternalException: INTERNAL Error: Failed to load metadata pointer (id 2626, idx 18, ptr 1297036692682705474)


Stack Trace:

/home/lc/Projects/EconomicsBusiness/.venv/lib/python3.13/site-packages/duckdb/duckdb.cpython-313-x86_64-linux-gnu.so(_ZN6duckdb9Exception6ToJSONENS_13ExceptionTypeERKSs+0x53) [0x7f47d995a5f3]
/home/lc/Projects/EconomicsBusiness/.venv/lib/python3.13/site-packages/duckdb/duckdb.cpython-313-x86_64-linux-gnu.so(_ZN6duckdb9ExceptionC1ENS_13ExceptionTypeERKSs+0x16) [0x7f47d995a626]
/home/lc/Projects/EconomicsBusiness/.venv/lib/python3.13/site-packages/duckdb/duckdb.cpython-313-x86_64-linux-gnu.so(_ZN6duckdb17InternalExceptionC1ERKSs+0x11) [0x7f47d995cc31]
/home/lc/Projects/EconomicsBusiness/.venv/lib/python3.13/site-packages/duckdb/duckdb.cpython-313-x86_64-linux-gnu.so(_ZN6duckdb17InternalExceptionC2IJljmEEERKSsDpT_+0x187) [0x7f47da5a1c67]
/home/lc/Projects/EconomicsBusiness/.venv/lib/python3.13/site-packages/duckdb/duckdb.cpython-313-x86_64-linux-gnu.so(+0xad9f0c) [